# V7_gold · A股 ETF 板块轮动策略 — 一键日/周操作指引

**目标**：一个 notebook 从 0 到 1 跑通 **V7_gold** 策略并输出：

1. **每日操作指引** — 今天该持有什么、下一个调仓日是什么时候、当前 gate 状态；
2. **下周持仓清单** — 本周五收盘信号决定，下周一开盘执行；
3. **交易指令列表**（相对上周的买/卖/加减仓）。

### 策略核心（V7_gold）

- **Leg A** 动量×拥挤惩罚（4w mom − 1.5·4w turn + 0.3·breadth），top-4 等权；
- **Leg G** 9 组主线轮动，top-3 组各挑 1 只领头羊；
- **Ensemble** 50/50 混合 Leg A 与 Leg G，每周最多 7 只 ETF；
- **Gate** 市场累计净值跌破 50 周 MA → **100% 切换黄金 ETF (159934.SZ)**；
- **Vol Target** 15% 年化，用 26 周 realized vol 反向缩放总暴露；
- **执行** 周五收盘信号，下周一开盘 + 单边 5bps 滑点。

### 依赖

```bash
pip install pandas numpy tushare pyarrow
export TUSHARE_TOKEN="your_token"   # Linux/macOS
$env:TUSHARE_TOKEN="your_token"       # Windows PowerShell
```

## 1 · Setup & Imports

In [1]:
import os, sys, time, json, warnings
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

HERE  = Path.cwd()
CACHE = HERE / 'data_cache'; CACHE.mkdir(parents=True, exist_ok=True)
OUT   = HERE / 'results';   OUT.mkdir(parents=True, exist_ok=True)

def say(s): print(f'[{time.strftime("%H:%M:%S")}] {s}', flush=True)
say(f'working dir: {HERE}')

[11:42:29] working dir: C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0


## 2 · Universe: 34 主题 ETF + 黄金 fallback


In [2]:
ETFS = {
    '半导体ETF':'512480.SH','消费电子ETF':'159779.SZ','软件ETF':'515230.SH',
    '通信ETF':'515880.SH','云计算ETF':'159890.SZ','AI_ETF':'515980.SH','游戏ETF':'159869.SZ',
    '光伏ETF':'159857.SZ','电池ETF':'159755.SZ','新能源车ETF':'515030.SH','电网设备ETF':'159326.SZ',
    '机器人ETF':'562500.SH','航空航天ETF':'159227.SZ','军工ETF':'512660.SH',
    '银行ETF':'512800.SH','证券ETF':'512880.SH','非银ETF':'159892.SZ',
    '酒ETF':'512690.SH','家电ETF':'561120.SH','食品ETF':'515170.SH',
    '医药ETF':'512010.SH','创新药ETF':'159992.SZ','医疗器械ETF':'159883.SZ',
    '有色ETF':'159980.SZ','煤炭ETF':'515220.SH','钢铁ETF':'515210.SH','石油ETF':'561360.SH',
    '化工ETF':'159870.SZ','稀土ETF':'159713.SZ','黄金ETF':'159934.SZ',
    '房地产ETF':'512200.SH','建材ETF':'516750.SH',
    '畜牧ETF':'159867.SZ','红利ETF':'515080.SH',
}
GROUPS = {
    '科技成长': ['半导体ETF','消费电子ETF','软件ETF','通信ETF','云计算ETF','AI_ETF','游戏ETF'],
    '新能源':   ['光伏ETF','电池ETF','新能源车ETF','电网设备ETF'],
    '高端制造': ['机器人ETF','航空航天ETF','军工ETF'],
    '大金融':   ['银行ETF','证券ETF','非银ETF'],
    '大消费':   ['酒ETF','家电ETF','食品ETF','医药ETF','创新药ETF','医疗器械ETF'],
    '周期资源': ['有色ETF','煤炭ETF','钢铁ETF','石油ETF','化工ETF','稀土ETF','黄金ETF'],
    '地产链':   ['房地产ETF','建材ETF'],
    '农业':     ['畜牧ETF'],
    '红利':     ['红利ETF'],
}
GOLD = '159934.SZ'

uni_df = pd.DataFrame([
    {'name':n, 'ts_code':c, 'group': next((g for g,m in GROUPS.items() if n in m), None)}
    for n,c in ETFS.items()
])
uni_df.to_csv(CACHE/'etf_universe.csv', index=False)
code2name  = dict(zip(uni_df['ts_code'], uni_df['name']))
code2group = dict(zip(uni_df['ts_code'], uni_df['group']))
say(f'Universe saved: {len(uni_df)} ETFs across {uni_df.group.nunique()} groups')
uni_df.head()

[11:42:29] Universe saved: 34 ETFs across 9 groups


,name,ts_code,group
0,半导体ETF,512480.SH,科技成长
1,消费电子ETF,159779.SZ,科技成长
2,软件ETF,515230.SH,科技成长
3,通信ETF,515880.SH,科技成长
4,云计算ETF,159890.SZ,科技成长


## 3 · 拉取 Tushare 数据（已缓存则跳过）

In [3]:
FD  = CACHE/'fund_daily_34.parquet'
FAJ = CACHE/'fund_adj_34.parquet'

def fetch_tushare():
    import tushare as ts
    token = os.environ.get('TUSHARE_TOKEN')
    if not token:
        raise RuntimeError('TUSHARE_TOKEN not set. Please export it first.')
    pro = ts.pro_api(token)
    end = time.strftime('%Y%m%d')

    if not FD.exists():
        rows = []
        for name, code in ETFS.items():
            for k in range(3):
                try:
                    df = pro.fund_daily(ts_code=code, start_date='20150101', end_date=end); break
                except Exception as e:
                    say(f'  retry {name} ({code}): {e}'); time.sleep(2+k)
            else:
                df = pd.DataFrame()
            if len(df)==0:
                say(f'  NO DATA {name} ({code})'); continue
            df['etf_name'] = name; rows.append(df)
            say(f'  daily  {name} ({code}): {len(df)} rows')
        fd = pd.concat(rows, ignore_index=True)
        fd['trade_date'] = pd.to_datetime(fd['trade_date'])
        fd.to_parquet(FD); say(f'saved {FD}')
    else:
        say(f'cached: {FD}')

    if not FAJ.exists():
        rows = []
        for name, code in ETFS.items():
            for k in range(3):
                try:
                    df = pro.fund_adj(ts_code=code, start_date='20150101', end_date=end); break
                except Exception as e:
                    time.sleep(2+k)
            else:
                df = pd.DataFrame()
            if len(df)==0: continue
            rows.append(df); say(f'  adj    {name} ({code}): {len(df)} rows')
        fa = pd.concat(rows, ignore_index=True)
        fa['trade_date'] = pd.to_datetime(fa['trade_date'])
        fa.to_parquet(FAJ); say(f'saved {FAJ}')
    else:
        say(f'cached: {FAJ}')

fetch_tushare()

[11:42:30] cached: C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0\data_cache\fund_daily_34.parquet


[11:42:30] cached: C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0\data_cache\fund_adj_34.parquet


## 4 · 构造周频面板（ISO-week Friday close）

In [4]:
fd = pd.read_parquet(FD)
fa = pd.read_parquet(FAJ)
df = fd.merge(fa, on=['ts_code','trade_date'], how='left')
df['close_adj'] = df['close'] * df['adj_factor'].fillna(1.0)
df = df.sort_values(['ts_code','trade_date']).reset_index(drop=True)
df['ret'] = df.groupby('ts_code')['close_adj'].pct_change()
df['turnover_proxy'] = df['amount']
df['year_week'] = df['trade_date'].dt.strftime('%G-%V')
df['gross'] = 1 + df['ret'].fillna(0)
df['cum']   = df.groupby('ts_code')['gross'].cumprod()

weekly = (df.groupby(['ts_code','etf_name','year_week'], sort=False)
            .agg(trade_week=('trade_date','last'),
                 cum_w=('cum','last'),
                 turnover_w=('turnover_proxy','mean'),
                 amount_w=('amount','sum'))
            .reset_index()
            .sort_values(['ts_code','trade_week']))
weekly['ret_w'] = weekly.groupby('ts_code')['cum_w'].pct_change()
weekly = weekly.dropna(subset=['ret_w']).copy()
weekly.to_parquet(CACHE/'etf_weekly.parquet')
say(f'weekly panel: {len(weekly):,} rows · {weekly.trade_week.nunique()} weeks · {weekly.ts_code.nunique()} ETFs')
say(f'sample: {weekly.trade_week.min().date()} → {weekly.trade_week.max().date()}')

daily_last = df.groupby('ts_code')['trade_date'].max().to_frame('last_daily')
say(f'latest daily bar:  {df.trade_date.max().date()}')

[11:42:30] weekly panel: 10,232 rows · 580 weeks · 34 ETFs


[11:42:30] sample: 2015-01-16 → 2026-04-22


[11:42:30] latest daily bar:  2026-04-22


## 5 · V7_gold 信号构造 + 回测

In [5]:
SAMPLE_START = '2019-01-01'
VOL_TARGET   = 0.15
VOL_LB       = 26
GATE_MA      = 50
DELAY        = 1
COST_BPS     = 5
MOM_W        = 4
TURN_W       = 4
LAMBDA       = 1.5
MU           = 0.3
TOP_N_A      = 4
TOP_K_GROUPS = 3
PER_GROUP    = 1
ELIG_WEEKS   = 12

W = weekly.copy()
W['trade_week'] = pd.to_datetime(W['trade_week'])
W = W[W['trade_week'] >= SAMPLE_START].sort_values(['trade_week','ts_code']).reset_index(drop=True)

ret  = W.pivot(index='trade_week', columns='ts_code', values='ret_w').sort_index()
turn = W.pivot(index='trade_week', columns='ts_code', values='turnover_w').sort_index()
cum  = (1 + ret.fillna(0)).cumprod()
age  = ret.notna().cumsum(); elig = age >= ELIG_WEEKS
cols = ret.columns

def zscore_cs(frame):
    df2 = frame.where(elig); m = df2.mean(axis=1); s = df2.std(axis=1)
    return df2.sub(m, axis=0).div(s, axis=0)

momX  = cum / cum.shift(MOM_W) - 1
turnX = turn.rolling(TURN_W, min_periods=max(2, TURN_W//2)).mean()
br_s  = ((cum > cum.rolling(20, min_periods=8).mean()).where(elig).sum(axis=1)
         / elig.sum(axis=1).replace(0, np.nan))
br_z  = (br_s - br_s.rolling(52, min_periods=20).mean()) / br_s.rolling(52, min_periods=20).std()
bread_df = pd.DataFrame({c: br_z for c in cols})
z_m, z_t = zscore_cs(momX), zscore_cs(turnX)

# --- Leg A ---
score_A = (z_m - LAMBDA*z_t + MU*bread_df).reindex(columns=cols).astype(float)
wA = np.zeros(score_A.shape)
sv = score_A.values; rv = ret.values; em = elig.values.astype(bool)
for ti in range(sv.shape[0]):
    s = sv[ti]; v = np.isfinite(s) & np.isfinite(rv[ti]) & em[ti]
    if v.sum() < TOP_N_A: continue
    sv_v = np.where(v, s, -np.inf)
    ix = np.argpartition(-sv_v, TOP_N_A)[:TOP_N_A]
    wA[ti, ix] = 1.0/TOP_N_A
w_A = pd.DataFrame(wA, index=ret.index, columns=cols)

# --- Leg G ---
group_names = sorted(set(uni_df['group'].dropna()))
gret = pd.DataFrame(0.0, index=ret.index, columns=group_names)
for g in group_names:
    codes_g = [c for c in uni_df[uni_df['group']==g]['ts_code'] if c in cols]
    gret[g] = ret[codes_g].where(elig[codes_g]).mean(axis=1)
gcum  = (1+gret.fillna(0)).cumprod()
gmom4 = gcum/gcum.shift(MOM_W) - 1

wG = np.zeros((len(ret), len(cols)))
for ti in range(len(ret)):
    gs = gmom4.iloc[ti].dropna()
    if len(gs) < TOP_K_GROUPS: continue
    tops = gs.nlargest(TOP_K_GROUPS).index.tolist()
    picks = []
    for g in tops:
        codes_g = [c for c in uni_df[uni_df['group']==g]['ts_code'] if c in cols]
        s = momX.iloc[ti][codes_g].dropna()
        s = s[[c for c in s.index if elig.iloc[ti][c]]]
        if len(s) < PER_GROUP: continue
        picks += s.nlargest(PER_GROUP).index.tolist()
    if not picks: continue
    nw = 1.0/len(picks)
    for c in picks: wG[ti, cols.get_loc(c)] = nw
w_G = pd.DataFrame(wG, index=ret.index, columns=cols)

# --- Ensemble + Gate + fallback ---
w_ens = 0.5*w_A + 0.5*w_G
mc   = cum.mean(axis=1)
mcma = mc.rolling(GATE_MA, min_periods=GATE_MA//2).mean()
gate_on = (mc > mcma).reindex(ret.index).fillna(False)
w = w_ens.copy()
off = ~gate_on
w.loc[off] = 0.0
gold_ok = elig[GOLD].fillna(False)
for t in w.index[off & gold_ok]:
    w.at[t, GOLD] = 1.0

# --- Vol target ---
pnl_raw = (w.shift(DELAY) * ret).sum(axis=1)
rv_w = pnl_raw.rolling(VOL_LB, min_periods=8).std()*np.sqrt(52)
scale = (VOL_TARGET / rv_w).clip(upper=1.0).fillna(0)
w_final = w.mul(scale.reindex(w.index), axis=0)

# --- Backtest ---
w_exec  = w_final.shift(DELAY)
pnl_g   = (w_exec * ret).sum(axis=1)
tov     = (w_exec.fillna(0).diff().abs().sum(axis=1)/2).fillna(0)
cost    = tov*(COST_BPS/1e4)
pnl_net = pnl_g - cost
say(f'backtest ok · {len(pnl_net)} weeks · {pnl_net.index[0].date()} → {pnl_net.index[-1].date()}')

[11:42:32] backtest ok · 377 weeks · 2019-01-04 → 2026-04-22


## 6 · 回测绩效（Full / IS / OOS）

In [6]:
def stats(pnl):
    r = pnl.dropna()
    if len(r) < 4: return {}
    yrs = len(r)/52.0
    ann = (1+r).prod()**(1/yrs)-1
    vol = r.std()*np.sqrt(52)
    sh  = ann/vol if vol>0 else np.nan
    eq  = (1+r).cumprod(); dd = (eq/eq.cummax()-1).min()
    cal = ann/abs(dd) if dd<0 else np.nan
    down = r[r<0]; dv = down.std()*np.sqrt(52) if len(down)>0 else np.nan
    srt = ann/dv if dv and dv>0 else np.nan
    return {'n_weeks':len(r),'ann_ret':ann,'vol':vol,'sharpe':sh,'sortino':srt,
            'max_dd':dd,'calmar':cal,'win_rate':(r>0).mean()}

is_mask  = pnl_net.index < pd.Timestamp('2024-01-01')
oos_mask = pnl_net.index >= pd.Timestamp('2024-01-01')
tbl = pd.DataFrame({
    'IS  (2019-2023)': stats(pnl_net[is_mask]),
    'OOS (2024-now)':  stats(pnl_net[oos_mask]),
    'Full':            stats(pnl_net),
}).T
for c in ['ann_ret','vol','max_dd','win_rate']:
    if c in tbl: tbl[c] = tbl[c].map(lambda x: f'{x*100:+.2f}%' if pd.notna(x) else '-')
for c in ['sharpe','sortino','calmar']:
    if c in tbl: tbl[c] = tbl[c].map(lambda x: f'{x:.2f}' if pd.notna(x) else '-')
tbl

,n_weeks,ann_ret,vol,sharpe,sortino,max_dd,calmar,win_rate
IS (2019-2023),258.0000,+26.07%,+14.05%,1.86,2.98,-8.89%,2.93,+60.85%
OOS (2024-now),119.0000,+34.16%,+16.78%,2.04,4.02,-7.59%,4.50,+59.66%
Full,377.0000,+28.57%,+14.95%,1.91,3.31,-8.89%,3.21,+60.48%


In [7]:
# 分年绩效
r = pnl_net.dropna(); r.index = pd.to_datetime(r.index)
py_rows = []
for yr, grp in r.groupby(r.index.year):
    if len(grp) < 10: continue
    ann = (1+grp).prod()**(52/len(grp))-1
    vol = grp.std()*np.sqrt(52)
    sh  = ann/vol if vol>0 else np.nan
    eq  = (1+grp).cumprod(); dd = (eq/eq.cummax()-1).min()
    py_rows.append({'year':yr,'ann_ret':f'{ann*100:+.2f}%','sharpe':f'{sh:.2f}',
                    'max_dd':f'{dd*100:+.2f}%','n_weeks':len(grp)})
py = pd.DataFrame(py_rows)
py.to_csv(OUT/'peryear.csv', index=False)
py

,year,ann_ret,sharpe,max_dd,n_weeks
0,2019,+25.79%,2.28,-5.59%,51
1,2020,+44.88%,2.56,-8.89%,52
2,2021,+30.04%,1.85,-6.19%,55
3,2022,+7.79%,0.81,-8.28%,50
4,2023,+23.58%,1.70,-4.96%,50
5,2024,+38.50%,2.11,-4.93%,51
6,2025,+35.92%,2.30,-5.75%,53
7,2026,+14.98%,0.91,-6.46%,15


## 7 · 📅 每日操作指引（Daily Playbook）

A 股周度调仓策略的日内行为非常简单——**绝大多数交易日就是持有不动**。以下给出明确的日历化动作表。

In [8]:
t_last      = ret.index[-1]
last_daily  = df['trade_date'].max()
today       = pd.Timestamp(datetime.now().date())

# 下一个调仓日：下周一（假设 t_last 是周五；如最后一条是当前周的周五，则下周一为 t_last+3）
def next_monday(d):
    d = pd.Timestamp(d)
    days_ahead = (0 - d.weekday()) % 7
    if days_ahead == 0: days_ahead = 7
    return d + timedelta(days=days_ahead)

next_rebal   = next_monday(t_last)
next_signal  = next_rebal - timedelta(days=3)   # 该调仓日前的周五
dow          = today.day_name()

print('═'*70)
print(f'  📅 今天          : {today.date()}  ({dow})')
print(f'  📊 最新周线收盘 : {t_last.date()}')
print(f'  📈 最新日线收盘 : {last_daily.date()}')
print(f'  🔁 下一次调仓   : {next_rebal.date()} (周一开盘)')
print(f'  💡 下一次信号   : {next_signal.date()} (周五收盘后)')
print('═'*70)

playbook = pd.DataFrame([
    {'星期':'周一','日内动作':'🛎️ 开盘按"下周持仓清单"调仓（市价/限价单，一次性买入或调整）','频率':'每周 1 次'},
    {'星期':'周二','日内动作':'✋ 持仓不动；可盘后看次日开盘回报率','频率':'被动持有'},
    {'星期':'周三','日内动作':'✋ 持仓不动','频率':'被动持有'},
    {'星期':'周四','日内动作':'✋ 持仓不动','频率':'被动持有'},
    {'星期':'周五','日内动作':'📝 收盘后运行本 notebook → 生成下周一的目标持仓清单','频率':'每周 1 次'},
    {'星期':'周末','日内动作':'🧠 review 回测指标、gate 状态、是否进入/退出 risk-off','频率':'可选'},
])
playbook

══════════════════════════════════════════════════════════════════════
  📅 今天          : 2026-04-23  (Thursday)
  📊 最新周线收盘 : 2026-04-22
  📈 最新日线收盘 : 2026-04-22
  🔁 下一次调仓   : 2026-04-27 (周一开盘)
  💡 下一次信号   : 2026-04-24 (周五收盘后)
══════════════════════════════════════════════════════════════════════


,星期,日内动作,频率
0,周一,"🛎️ 开盘按""下周持仓清单""调仓（市价/限价单，一次性买入或调整）",每周 1 次
1,周二,✋ 持仓不动；可盘后看次日开盘回报率,被动持有
2,周三,✋ 持仓不动,被动持有
3,周四,✋ 持仓不动,被动持有
4,周五,📝 收盘后运行本 notebook → 生成下周一的目标持仓清单,每周 1 次
5,周末,🧠 review 回测指标、gate 状态、是否进入/退出 risk-off,可选


In [9]:
# 当前 Gate 状态与 risk 模式
gate_now = bool(gate_on.iloc[-1])
rv_now   = rv_w.iloc[-1]
scale_now= scale.iloc[-1]

mode = '🟢 RISK-ON (主题 ETF 轮动)' if gate_now else '🔴 RISK-OFF (100% 黄金 ETF)'
print('═'*70)
print(f'  当前市场状态 (截至 {t_last.date()})')
print('─'*70)
print(f'  市场累计净值       : {mc.iloc[-1]:.4f}')
print(f'  50w MA             : {mcma.iloc[-1]:.4f}')
print(f'  Gate               : {"ON  (mkt > MA50)" if gate_now else "OFF (mkt <= MA50)"}')
print(f'  Mode               : {mode}')
print(f'  策略 26w 已实现波动 : {rv_now*100:.2f}% (年化)')
print(f'  Vol target         : {VOL_TARGET*100:.2f}%  →  scale = {scale_now:.3f}')
print(f'  当前总暴露上限      : {scale_now*100:.1f}% (剩余 {(1-scale_now)*100:.1f}% 现金)')
print('═'*70)

══════════════════════════════════════════════════════════════════════
  当前市场状态 (截至 2026-04-22)
──────────────────────────────────────────────────────────────────────
  市场累计净值       : 1.5702
  50w MA             : 1.3826
  Gate               : ON  (mkt > MA50)
  Mode               : 🟢 RISK-ON (主题 ETF 轮动)
  策略 26w 已实现波动 : 27.54% (年化)
  Vol target         : 15.00%  →  scale = 0.545
  当前总暴露上限      : 54.5% (剩余 45.5% 现金)
══════════════════════════════════════════════════════════════════════


## 8 · 📋 下周持仓清单（Next Week Holdings）

以 **最近一个周五收盘** 的信号，生成 **下周一开盘** 应调至的目标持仓。

In [10]:
nz = w_final.iloc[-1]
nz = nz[nz > 1e-5].sort_values(ascending=False)

next_hold = pd.DataFrame({
    'ts_code': nz.index,
    'ETF名称':  [code2name.get(c,'?')  for c in nz.index],
    '板块':     [code2group.get(c,'?') for c in nz.index],
    'w_A(50%)': [f'{w_A.iloc[-1].get(c,0)*0.5*scale_now*100:5.2f}%' for c in nz.index],
    'w_G(50%)': [f'{w_G.iloc[-1].get(c,0)*0.5*scale_now*100:5.2f}%' for c in nz.index],
    '目标权重': [f'{v*100:6.2f}%' for v in nz.values],
    'scale':   [f'{scale_now:.3f}'] * len(nz),
})

print('═'*78)
print(f'  📋 下周一 ({next_rebal.date()}) 开盘目标持仓')
print(f'  📊 信号截止 : {t_last.date()} 周五收盘')
print(f'  💰 总仓位   : {nz.sum()*100:.1f}%   ·   现金: {(1-nz.sum())*100:.1f}%')
print(f'  ⚙️  Mode     : {"RISK-ON" if gate_now else "RISK-OFF (100% gold)"}')
print('═'*78)
display(next_hold.reset_index(drop=True))

next_hold.to_csv(OUT/f'next_holdings_{t_last.date()}.csv', index=False)
say(f'saved: {OUT / ("next_holdings_"+str(t_last.date())+".csv")}')

══════════════════════════════════════════════════════════════════════════════
  📋 下周一 (2026-04-27) 开盘目标持仓
  📊 信号截止 : 2026-04-22 周五收盘
  💰 总仓位   : 54.5%   ·   现金: 45.5%
  ⚙️  Mode     : RISK-ON
══════════════════════════════════════════════════════════════════════════════


,ts_code,ETF名称,板块,w_A(50%),w_G(50%),目标权重,scale
0,159326.SZ,电网设备ETF,新能源,0.00%,9.08%,9.08%,0.545
1,562500.SH,机器人ETF,高端制造,0.00%,9.08%,9.08%,0.545
2,515880.SH,通信ETF,科技成长,0.00%,9.08%,9.08%,0.545
3,159779.SZ,消费电子ETF,科技成长,6.81%,0.00%,6.81%,0.545
4,159713.SZ,稀土ETF,周期资源,6.81%,0.00%,6.81%,0.545
5,159890.SZ,云计算ETF,科技成长,6.81%,0.00%,6.81%,0.545
6,515980.SH,AI_ETF,科技成长,6.81%,0.00%,6.81%,0.545


[11:42:32] saved: C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0\results\next_holdings_2026-04-22.csv


## 9 · 🔄 交易指令清单（Trade List vs 上周）

In [11]:
w_now  = w_final.iloc[-1]
w_prev = w_final.iloc[-2]
delta  = (w_now - w_prev)
active = delta[delta.abs() > 1e-4].sort_values()

trade_rows = []
for code, d in active.items():
    side = '🟢 BUY' if d>0 else '🔴 SELL'
    trade_rows.append({
        '方向': side,
        'ts_code': code,
        'ETF名称': code2name.get(code,'?'),
        '板块':    code2group.get(code,'?'),
        '上周权重': f'{w_prev.get(code,0)*100:+6.2f}%',
        '本周权重': f'{w_now.get(code,0)*100:+6.2f}%',
        'Δ权重':   f'{d*100:+6.2f}%',
    })
trade_df = pd.DataFrame(trade_rows)

tov_this = delta.abs().sum()/2
print('═'*78)
print(f'  🔄 调仓交易指令  ({ret.index[-2].date()} → {t_last.date()})')
print(f'  单边换手率     : {tov_this*100:.1f}%')
print('═'*78)
if trade_df.empty:
    print('  ✅ 无需调仓（本周目标持仓与上周相同）')
else:
    display(trade_df)
trade_df.to_csv(OUT/f'trades_{t_last.date()}.csv', index=False)

══════════════════════════════════════════════════════════════════════════════
  🔄 调仓交易指令  (2026-04-17 → 2026-04-22)
  单边换手率     : 9.1%
══════════════════════════════════════════════════════════════════════════════


,方向,ts_code,ETF名称,板块,上周权重,本周权重,Δ权重
0,🔴 SELL,515030.SH,新能源车ETF,新能源,+9.05%,+0.00%,-9.05%
1,🟢 BUY,159713.SZ,稀土ETF,周期资源,+6.79%,+6.81%,+0.02%
2,🟢 BUY,159890.SZ,云计算ETF,科技成长,+6.79%,+6.81%,+0.02%
3,🟢 BUY,159779.SZ,消费电子ETF,科技成长,+6.79%,+6.81%,+0.02%
4,🟢 BUY,515980.SH,AI_ETF,科技成长,+6.79%,+6.81%,+0.02%
5,🟢 BUY,515880.SH,通信ETF,科技成长,+9.05%,+9.08%,+0.03%
6,🟢 BUY,562500.SH,机器人ETF,高端制造,+9.05%,+9.08%,+0.03%
7,🟢 BUY,159326.SZ,电网设备ETF,新能源,+0.00%,+9.08%,+9.08%


## 10 · 🔍 信号穿透（Why these picks?）

In [12]:
# Leg A top-4 排名
tab_A = pd.DataFrame({
    'ETF':     [code2name.get(c,'?')  for c in cols],
    '板块':    [code2group.get(c,'?') for c in cols],
    'mom_4w':  (momX.iloc[-1].values*100),
    'turn_4w': turnX.iloc[-1].values,
    'z_mom':   z_m.iloc[-1].values,
    'z_turn':  z_t.iloc[-1].values,
    'score_A': score_A.iloc[-1].values,
    'eligible':elig.iloc[-1].values,
}, index=cols)
tab_A = tab_A[tab_A['eligible']].sort_values('score_A', ascending=False)
tab_A['mom_4w']  = tab_A['mom_4w'].map(lambda x: f'{x:+6.2f}%')
tab_A['z_mom']   = tab_A['z_mom'].map(lambda x: f'{x:+.2f}')
tab_A['z_turn']  = tab_A['z_turn'].map(lambda x: f'{x:+.2f}')
tab_A['score_A'] = tab_A['score_A'].map(lambda x: f'{x:+.3f}')
print(f'🎯 Leg A · penalized momentum 排名 (top-10, 其中前 4 被选中 25% 等权):')
display(tab_A.head(10))

🎯 Leg A · penalized momentum 排名 (top-10, 其中前 4 被选中 25% 等权):


,ETF,板块,mom_4w,turn_4w,z_mom,z_turn,score_A,eligible
ts_code,,,,,,,,
159779.SZ,消费电子ETF,科技成长,+21.25%,"9,653.6596",+2.07,-1.00,+3.417,True
159890.SZ,云计算ETF,科技成长,+12.86%,"38,467.4530",+0.96,-0.94,+2.220,True
515980.SH,AI_ETF,科技成长,+19.84%,"378,593.9102",+1.88,-0.26,+2.126,True
159713.SZ,稀土ETF,周期资源,+8.37%,"86,862.0412",+0.37,-0.85,+1.482,True
561120.SH,家电ETF,大消费,+4.08%,"10,648.1801",-0.20,-1.00,+1.142,True
515230.SH,软件ETF,科技成长,+6.28%,"132,839.0686",+0.09,-0.75,+1.068,True
512660.SH,军工ETF,高端制造,+9.57%,"389,644.1534",+0.52,-0.24,+0.734,True
515030.SH,新能源车ETF,新能源,+7.34%,"295,648.8295",+0.23,-0.43,+0.721,True
159227.SZ,航空航天ETF,高端制造,+7.25%,"301,757.7931",+0.22,-0.42,+0.690,True


In [13]:
# Leg G 9-组排名
gm_now = gmom4.iloc[-1].sort_values(ascending=False)
picks_G = []
per_group = []
top3 = gm_now.head(TOP_K_GROUPS).index.tolist()
for g in gm_now.index:
    mark = '⭐ PICKED' if g in top3 else ''
    per_group.append({'组': g, '组内4w动量': f'{gm_now[g]*100:+6.2f}%', '状态': mark})
print('🎯 Leg G · 9 组主线 4w 动量 (top-3 组各挑 1 只领头羊):')
display(pd.DataFrame(per_group))

leaders = []
for g in top3:
    codes_g = [c for c in uni_df[uni_df['group']==g]['ts_code'] if c in cols]
    s = momX.iloc[-1][codes_g].dropna()
    s = s[[c for c in s.index if elig.iloc[-1][c]]]
    if len(s)==0: continue
    leader = s.idxmax()
    leaders.append({'组': g, '领头羊': code2name.get(leader,'?'), 'ts_code': leader,
                    '4w动量': f'{s[leader]*100:+6.2f}%'})
print('\n🎯 top-3 组各自领头羊:')
display(pd.DataFrame(leaders))

🎯 Leg G · 9 组主线 4w 动量 (top-3 组各挑 1 只领头羊):


,组,组内4w动量,状态
0,科技成长,+15.41%,⭐ PICKED
1,高端制造,+8.90%,⭐ PICKED
2,新能源,+4.91%,⭐ PICKED
3,大金融,+3.16%,
4,周期资源,+2.69%,
5,地产链,+1.02%,
6,农业,+0.97%,
7,大消费,+0.67%,
8,红利,+0.19%,



🎯 top-3 组各自领头羊:


,组,领头羊,ts_code,4w动量
0,科技成长,通信ETF,515880.SH,+32.18%
1,高端制造,机器人ETF,562500.SH,+9.89%
2,新能源,电网设备ETF,159326.SZ,+7.41%


## 10.5 · 📈 NAV vs CSI 300 + 超额 + Drawdown

拉取沪深 300 作为 benchmark，绘制三联图：
1. **NAV 对比**：策略 vs 沪深 300（从 2019-01 起，均归一到 1.0）
2. **超额净值**：策略 NAV / CSI300 NAV
3. **Drawdown 对比**：两条曲线的回撤

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['SimHei','Microsoft YaHei','DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 拉取沪深 300 日线
import tushare as ts
pro = ts.pro_api(os.environ['TUSHARE_TOKEN'])
bm_daily = pro.index_daily(ts_code='000300.SH', start_date='20181201',
                            end_date=time.strftime('%Y%m%d'))
bm_daily['trade_date'] = pd.to_datetime(bm_daily['trade_date'])
bm_daily = bm_daily.sort_values('trade_date').reset_index(drop=True)
bm_daily['ret'] = bm_daily['close'].pct_change()
bm_daily['year_week'] = bm_daily['trade_date'].dt.strftime('%G-%V')
bm_w = (bm_daily.groupby('year_week')
        .agg(trade_week=('trade_date','last'), close_w=('close','last'))
        .reset_index().sort_values('trade_week'))
bm_w['ret_w'] = bm_w['close_w'].pct_change()
bm_ret = bm_w.set_index('trade_week')['ret_w'].dropna()
bm_ret.index = pd.to_datetime(bm_ret.index)
say(f'CSI 300 weekly returns: {len(bm_ret)} weeks · {bm_ret.index[0].date()} → {bm_ret.index[-1].date()}')

# 对齐周度
common_idx = pnl_net.index.intersection(bm_ret.index)
r_strat = pnl_net.reindex(common_idx).fillna(0)
r_bm    = bm_ret.reindex(common_idx).fillna(0)

nav_strat = (1 + r_strat).cumprod()
nav_bm    = (1 + r_bm).cumprod()
excess    = nav_strat / nav_bm

dd_strat = nav_strat / nav_strat.cummax() - 1
dd_bm    = nav_bm    / nav_bm.cummax()    - 1
dd_exc   = excess    / excess.cummax()    - 1

# Stats
def _s(r):
    y = len(r)/52.0
    a = (1+r).prod()**(1/y) - 1
    v = r.std()*np.sqrt(52)
    return a, v, a/v if v>0 else np.nan, (((1+r).cumprod()/(1+r).cumprod().cummax())-1).min()

a_s,v_s,sh_s,dd_s = _s(r_strat)
a_b,v_b,sh_b,dd_b = _s(r_bm)
ex_ret = r_strat - r_bm
a_e,v_e,sh_e,dd_e = _s(ex_ret)

print('═'*70)
print(f'  策略 (V7_gold)     : AnnRet {a_s*100:+6.2f}%   Sharpe {sh_s:5.2f}   MaxDD {dd_s*100:+6.2f}%')
print(f'  沪深300 (benchmark) : AnnRet {a_b*100:+6.2f}%   Sharpe {sh_b:5.2f}   MaxDD {dd_b*100:+6.2f}%')
print(f'  超额 (策略-沪深300) : AnnRet {a_e*100:+6.2f}%   Sharpe {sh_e:5.2f}   MaxDD {dd_e*100:+6.2f}%')
print('═'*70)

# 三联图
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True, gridspec_kw={'height_ratios':[3,2,2]})

ax = axes[0]
ax.plot(nav_strat.index, nav_strat.values, color='#C0392B', lw=2.0, label=f'V7_gold (Ann {a_s*100:+.1f}%, Sh {sh_s:.2f})')
ax.plot(nav_bm.index,    nav_bm.values,    color='#2C3E50', lw=1.6, label=f'CSI 300 (Ann {a_b*100:+.1f}%, Sh {sh_b:.2f})', linestyle='--')
ax.set_yscale('log')
ax.set_ylabel('NAV (log scale)')
ax.set_title(f'V7_gold vs CSI 300 · Full sample {common_idx[0].date()} → {common_idx[-1].date()}', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(1.0, color='gray', lw=0.8, alpha=0.5)

ax = axes[1]
ax.plot(excess.index, excess.values, color='#27AE60', lw=1.8, label=f'Excess NAV (Ann {a_e*100:+.1f}%, Sh {sh_e:.2f})')
ax.fill_between(excess.index, 1.0, excess.values, where=excess.values>=1, color='#27AE60', alpha=0.15)
ax.fill_between(excess.index, 1.0, excess.values, where=excess.values<1,  color='#C0392B', alpha=0.15)
ax.axhline(1.0, color='gray', lw=0.8)
ax.set_ylabel('Excess NAV (策略/CSI300)')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.fill_between(dd_strat.index, 0, dd_strat.values*100, color='#C0392B', alpha=0.55, label=f'V7_gold DD (min {dd_s*100:+.2f}%)')
ax.fill_between(dd_bm.index,    0, dd_bm.values*100,    color='#2C3E50', alpha=0.35, label=f'CSI 300 DD (min {dd_b*100:+.2f}%)')
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', lw=0.6)

plt.tight_layout()
fig_path = OUT / 'nav_vs_benchmark.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
say(f'figure saved: {fig_path}')

# Save comparison CSV
cmp = pd.DataFrame({'nav_v7_gold':nav_strat, 'nav_csi300':nav_bm,
                    'excess':excess, 'dd_strat':dd_strat, 'dd_bm':dd_bm})
cmp.to_csv(OUT/'nav_vs_benchmark.csv')
say(f'data saved: {OUT/"nav_vs_benchmark.csv"}')

[11:42:35] CSI 300 weekly returns: 377 weeks · 2018-12-14 → 2026-04-22


══════════════════════════════════════════════════════════════════════
  策略 (V7_gold)     : AnnRet +28.75%   Sharpe  1.92   MaxDD  -8.89%
  沪深300 (benchmark) : AnnRet  +6.70%   Sharpe  0.37   MaxDD -45.60%
  超额 (策略-沪深300) : AnnRet +18.24%   Sharpe  1.04   MaxDD -28.15%
══════════════════════════════════════════════════════════════════════


[11:42:35] figure saved: C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0\results\nav_vs_benchmark.png


[11:42:35] data saved: C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0\results\nav_vs_benchmark.csv


## 11 · 💾 输出所有文件


In [15]:
# 权益曲线 + 周频 PnL
pnl_net.to_frame('pnl_net').to_csv(OUT/'pnl.csv')
(1+pnl_net.fillna(0)).cumprod().to_frame('equity').to_csv(OUT/'equity_curve.csv')

# holdings 历史
total = w_exec.fillna(0).sum(axis=0)
avg = total/len(w_exec)
h_rows = []
for c in total.index:
    if total[c]==0: continue
    h_rows.append({'ts_code':c,'etf_name':code2name.get(c,'?'),'group':code2group.get(c,'?'),
                   'avg_weight':avg[c],'weeks_held':int((w_exec[c]>0).sum()),
                   'pct_weeks_held':float((w_exec[c]>0).mean())})
hist_hold = pd.DataFrame(h_rows).sort_values('avg_weight', ascending=False).head(25).reset_index(drop=True)
hist_hold.to_csv(OUT/'holdings.csv', index=False)

# metrics json
metrics = {
    'spec':'V7_gold',
    'sample':{'start':str(ret.index[0].date()), 'end':str(ret.index[-1].date()), 'n_weeks':len(ret)},
    'full':stats(pnl_net),
    'is':  stats(pnl_net[is_mask]),
    'oos': stats(pnl_net[oos_mask]),
    'gate_on_rate': float(gate_on.mean()),
    'next_rebalance_date': str(next_rebal.date()),
    'latest_signal_date':  str(t_last.date()),
    'mode_now': 'RISK-ON' if gate_now else 'RISK-OFF (gold)',
    'scale_now': float(scale_now),
}
with open(OUT/'metrics.json','w',encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False, default=str)

print('═'*70)
print(f'  ✅ 所有产出保存在 {OUT}')
print('─'*70)
for p in sorted(OUT.glob('*')):
    if p.is_file(): print(f'    {p.name:40s}  ({p.stat().st_size:,} bytes)')
print('═'*70)

══════════════════════════════════════════════════════════════════════
  ✅ 所有产出保存在 C:\Users\Hu\Downloads\Factor_Zoo-main\deploy\A-Share-ETF-Rotation-Strategy-2.0\results
──────────────────────────────────────────────────────────────────────
    drawdowns.csv                             (333 bytes)
    equity_curve.csv                          (11,354 bytes)
    holdings.csv                              (1,980 bytes)
    metrics.json                              (1,157 bytes)
    monthly.csv                               (4,558 bytes)
    nav_vs_benchmark.csv                      (38,560 bytes)
    nav_vs_benchmark.png                      (222,261 bytes)
    next_holdings_2026-04-22.csv              (509 bytes)
    peryear.csv                               (267 bytes)
    pnl.csv                                   (12,361 bytes)
    sortino_comparison.json                   (2,421 bytes)
    trades_2026-04-22.csv                     (620 bytes)
    v3_v7_full_stats.json                 

## ✅ 完成

- **每日**：查 cell 7（Daily Playbook）知道今天该做什么；
- **每周五收盘**：重跑本 notebook，cell 8 即为下周一开盘目标持仓；
- **调仓**：按 cell 9 的 BUY/SELL 清单执行。

> ⚠️ 本 notebook 仅供量化研究参考，不构成投资建议。实盘请自行做好风控与资金管理。